# Notebook to define the gene set, promoter boundaries, and gene body boundaries

We will read in the MNase-seq bam files collect the promoter and gene body fragments then compute

- Read in the SGD genes
- Read in the Park TSS dataset
- Filter out non dubious genes
- Define promoter region (300 bp to TSS)
- Define gene body boundary (TSS to 500 bp upstream)


In [75]:
# Preamble, notebook setup and imports

%load_ext autoreload
%autoreload 2
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [76]:
from src.sgd import read_sgd_genes, read_park_TSS_PAS

In [77]:
from src.read_bam import _fromRoman

# Read the genes from the SGD saccCer3 R64 gff file, converting chromosome names to the more iterable
# integers, skipping non nuclear chromosomes
genes = read_sgd_genes()
chroms = genes['chr'].str.replace('chr', '').map(_fromRoman)
genes['chr'] = chroms.astype(int)
genes = genes[genes['chr'] > 0]

# Read Park TSS annotations
park_TSS_PAS = read_park_TSS_PAS()
genes = genes.join(park_TSS_PAS)

# Set the TSS to the ORF start if the TSS is not in the Park data
genes.loc[(genes['TSS'].isna()) & (genes['strand'] == '+'), 'TSS'] = genes['start']
genes.loc[(genes['TSS'].isna()) & (genes['strand'] == '-'), 'TSS'] = genes['stop']
genes['TSS'] = genes['TSS'].astype(int)

# Remove dubious genes
nondub_genes = genes[genes['classification'] != 'Dubious'].copy()
nondub_genes.head(5)


,gene,chr,cat,start,stop,strand,classification,length,TSS,PAS,manually_curated
orf_name,,,,,,,,,,,
YAL068C,PAU8,1,gene,1807,2169,-,Verified,362,2169,NaN,NaN
YAL067W-A,None,1,gene,2480,2707,+,Uncharacterized,227,2480,NaN,NaN
YAL067C,SEO1,1,gene,7235,9016,-,Verified,1781,9016,NaN,NaN
YAL065C,None,1,gene,11565,11951,-,Uncharacterized,386,11951,NaN,NaN
YAL064W-B,None,1,gene,12046,12426,+,Uncharacterized,380,12046,NaN,NaN


In [90]:
# Define promoter
promoter_span = [-300, 0]

# Define gene body boundary
gene_body_span = [0, 500]

TSSes = nondub_genes['TSS']

is_watson = nondub_genes['strand'] == '+'

# ----------- Promoter boundary --------------

nondub_genes.loc[is_watson, 'promoter_start'] = TSSes+promoter_span[0]
nondub_genes.loc[is_watson, 'promoter_end'] = TSSes

nondub_genes.loc[~is_watson, 'promoter_start'] = TSSes
nondub_genes.loc[~is_watson, 'promoter_end'] = TSSes-promoter_span[0]

# ----------- Gene body boundary --------------

nondub_genes.loc[is_watson, 'gene_body_start'] = TSSes
nondub_genes.loc[is_watson, 'gene_body_end'] = TSSes+gene_body_span[1]

nondub_genes.loc[~is_watson, 'gene_body_start'] = TSSes-gene_body_span[1]
nondub_genes.loc[~is_watson, 'gene_body_end'] = TSSes

# Set the gene name to orf name if dne
nondub_genes.loc[nondub_genes['gene'].isna(), 'gene'] = nondub_genes[nondub_genes['gene'].isna()].index


In [91]:
nondub_genes.to_csv('data/geneset_nondub_w_prom_genebodies.csv')

In [92]:
nondub_genes

,gene,chr,cat,start,stop,strand,classification,length,TSS,PAS,manually_curated,promoter_start,promoter_end,gene_body_start,gene_body_end
orf_name,,,,,,,,,,,,,,,
YAL068C,PAU8,1,gene,1807,2169,-,Verified,362,2169,NaN,NaN,2169.0,2469.0,1669.0,2169.0
YAL067W-A,YAL067W-A,1,gene,2480,2707,+,Uncharacterized,227,2480,NaN,NaN,2180.0,2480.0,2480.0,2980.0
YAL067C,SEO1,1,gene,7235,9016,-,Verified,1781,9016,NaN,NaN,9016.0,9316.0,8516.0,9016.0
YAL065C,YAL065C,1,gene,11565,11951,-,Uncharacterized,386,11951,NaN,NaN,11951.0,12251.0,11451.0,11951.0
YAL064W-B,YAL064W-B,1,gene,12046,12426,+,Uncharacterized,380,12046,NaN,NaN,11746.0,12046.0,12046.0,12546.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YPR200C,ARR2,16,gene,939279,939671,-,Verified,392,939779,939195.0,False,939779.0,940079.0,939279.0,939779.0
YPR201W,ARR3,16,gene,939922,941136,+,Verified,1214,939858,NaN,False,939558.0,939858.0,939858.0,940358.0
YPR202W,YPR202W,16,gene,943032,943896,+,Uncharacterized,864,942768,NaN,False,942468.0,942768.0,942768.0,943268.0
